# 16. Dataset cleaning and filename audit

**Purpose:** back up the `Dataset_Multisite` directory, audit filename-label typos, apply the corrected rename list, and record the cleaning/verification statistics.

**Important — this notebook modifies files in Google Drive.** The later cells rename files and remove duplicates or other files according to the documented cleaning rules. Before using `Runtime > Run all`:

1. Confirm that `/content/drive/MyDrive/Dataset_Multisite` is the intended dataset folder.
2. Confirm that Google Drive has enough free space for a complete timestamped backup. The first cell uses `shutil.copytree` to create that backup.
3. Run the backup cell first and verify that the backup contains `Palm`, `Nail`, and `Conjunctiva`. Do not proceed if the backup is incomplete.
4. Review the generated rename list, especially `daftar_rename_typo_FIXED.csv`, before allowing the cleaning cell to modify the source folder.
5. Keep the backup until the cleaned dataset and the final leakage-audit results have been checked.

**Inputs:** `Dataset_Multisite/{Palm,Nail,Conjunctiva}/` in Google Drive.

**Outputs:** cleaning summaries and audit files in `hasil_revisi_v2/`, including `stats_pembersihan.json`, `stats_pembersihan_v2.json`, and `laporan_pembersihan_drive.json`.

Do not upload raw pediatric images, filenames that identify participants, or subject-level identifiers to this repository.


In [ ]:
# CELL 1: BACKUP DATASET (WAJIB SEBELUM PEMBERSIHAN)
# =============================================================================
# Membuat backup lengkap folder Dataset_Multisite sebelum dimodifikasi.
# Backup ini JANGAN dihapus sampai Anda yakin dataset bersih sudah benar.
# =============================================================================

import os
import shutil
from datetime import datetime

BASE = '/content/drive/MyDrive/Dataset_Multisite'
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP = f'/content/drive/MyDrive/Dataset_Multisite_BACKUP_{timestamp}'

print("=" * 70)
print("BACKUP DATASET (WAJIB SEBELUM PEMBERSIHAN)")
print("=" * 70)
print(f"Folder asli:  {BASE}")
print(f"Folder backup: {BACKUP}")
print()

# Cek apakah folder asli ada
if not os.path.exists(BASE):
    raise FileNotFoundError(f"Folder dataset tidak ditemukan: {BASE}")

# Hitung ukuran total sebelum backup
total_size = 0
total_files = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if os.path.exists(site_path):
        for f in os.listdir(site_path):
            fp = os.path.join(site_path, f)
            if os.path.isfile(fp):
                total_size += os.path.getsize(fp)
                total_files += 1

print(f"Total file yang akan di-backup: {total_files}")
print(f"Total ukuran: {total_size / (1024*1024):.1f} MB")
print()
print("Memulai backup... (ini bisa memakan waktu 5-10 menit)")
print()

# Lakukan backup
shutil.copytree(BASE, BACKUP)

# Verifikasi backup
backup_files = 0
backup_size = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BACKUP, site)
    if os.path.exists(site_path):
        for f in os.listdir(site_path):
            fp = os.path.join(site_path, f)
            if os.path.isfile(fp):
                backup_files += 1
                backup_size += os.path.getsize(fp)

print(f"✓ Backup selesai!")
print(f"  File di backup: {backup_files}")
print(f"  Ukuran backup: {backup_size / (1024*1024):.1f} MB")
print()
print(f"⚠ BACKUP TERSIMPAN DI: {BACKUP}")
print(f"  JANGAN HAPUS folder ini sampai Anda yakin dataset bersih sudah benar!")
print(f"  Jika ada masalah, Anda bisa restore dari backup ini.")
print()

# Verifikasi jumlah file sama
if backup_files == total_files:
    print("✓ Verifikasi: jumlah file sama (backup lengkap)")
else:
    print(f"⚠ PERINGATAN: jumlah file berbeda!")
    print(f"  Asli: {total_files}, Backup: {backup_files}")
    print(f"  Jangan lanjut ke Cell 2 sebelum ini diperbaiki!")

In [ ]:
# CELL 2 (FIXED v2): IDENTIFIKASI SEMUA TYPO DI DRIVE
# =============================================================================

import os
import re
import difflib
import pandas as pd
from datetime import datetime

BASE = '/content/drive/MyDrive/Dataset_Multisite'
OUT = '/content/drive/MyDrive/hasil_revisi_v2'
os.makedirs(OUT, exist_ok=True)

TYPO_THRESHOLD = 0.75


def extract_label_text(name):
    """Ekstrak teks label dari nama file (lowercase, tanpa hyphen/spasi)."""
    base = name.rsplit('.', 1)[0] if '.' in name else name
    base = re.sub(r'\s*\(\d+\)\s*$', '', base)
    match = re.match(r'^([A-Za-z\-]+)', base)
    if not match:
        return ''
    return match.group(1).lower().replace('-', '').replace(' ', '')


def similarity(a, b):
    """Hitung rasio kemiripan antara dua string."""
    return difflib.SequenceMatcher(None, a, b).ratio()


def detect_typo(filename):
    """Deteksi apakah file punya typo di nama."""
    text = extract_label_text(filename)
    if len(text) < 3:
        return False, None, None

    # Cek Non-Anemic (lebih dulu karena lebih panjang)
    target_non = 'nonanemic'
    if text.startswith(target_non):
        return False, 'nonanemic', None
    n = min(len(text), len(target_non))
    if similarity(text[:n], target_non[:n]) >= TYPO_THRESHOLD:
        return True, 'nonanemic', text[:n]

    # Cek Anemic
    target_an = 'anemic'
    if text.startswith(target_an):
        return False, 'anemic', None
    n = min(len(text), len(target_an))
    if similarity(text[:n], target_an[:n]) >= TYPO_THRESHOLD:
        return True, 'anemic', text[:n]

    return False, None, None


def generate_correct_name_FIXED(filename, correct_label):
    """
    VERSI PERBAIKAN: Hanya ganti kata typo pertama (alpha saja),
    sisanya (termasuk "-fn-", "-FN-", "-Fin-", nomor, suffix) tetap utuh.

    Contoh BENAR:
      'Anmeic-fn-0001 1.png'     → 'Anemic-fn-0001 1.png'      ✓
      'Anmeic-fn-0011 (2).png'   → 'Anemic-fn-0011 (2).png'    ✓
      'Anrmic-fn-001.png'        → 'Anemic-fn-001.png'         ✓
      'Non-anmeic-Fin-001.png'   → 'Non-anemic-Fin-001.png'    ✓
    """
    # Pisahkan ekstensi
    if '.' in filename:
        name, ext = filename.rsplit('.', 1)
        ext = '.' + ext
    else:
        name, ext = filename, ''

    # Hapus penanda salinan bernomor: "name (1)" → "name"
    numbered_match = re.search(r'\s*\(\d+\)\s*$', name)
    numbered_suffix = ''
    if numbered_match:
        numbered_suffix = numbered_match.group(0)
        name = name[:numbered_match.start()]

    # Split berdasarkan hyphen
    parts = name.split('-')

    if not parts:
        return filename

    # Ambil kata alfabetik pertama dari part pertama
    first_part = parts[0]
    alpha_match = re.match(r'^([A-Za-z]+)(.*)$', first_part)

    if not alpha_match:
        return filename

    word = alpha_match.group(1)
    rest_of_part = alpha_match.group(2)
    word_lower = word.lower()

    if correct_label == 'anemic':
        # Preserve case: jika "Anmeic" → "Anemic", jika "anmeic" → "anemic"
        if word[0].isupper():
            parts[0] = 'Anemic' + rest_of_part
        else:
            parts[0] = 'anemic' + rest_of_part

    elif correct_label == 'nonanemic':
        if word_lower == 'non':
            # Format: "Non-anmeic-..." → ganti part kedua
            if len(parts) > 1:
                second_part = parts[1]
                alpha2 = re.match(r'^([A-Za-z]+)(.*)$', second_part)
                if alpha2:
                    word2 = alpha2.group(1)
                    rest2 = alpha2.group(2)
                    if word2[0].isupper():
                        parts[1] = 'Anemic' + rest2
                    else:
                        parts[1] = 'anemic' + rest2
        else:
            # Format: "Nonanmeic-..." (satu kata)
            if word[0].isupper():
                parts[0] = 'Nonanemic' + rest_of_part
            else:
                parts[0] = 'nonanemic' + rest_of_part

    # Rekonstruksi nama
    new_name = '-'.join(parts) + numbered_suffix + ext
    return new_name


# =============================================================================
# TEST FUNGSI
# =============================================================================

print("=" * 70)
print("TEST FUNGSI generate_correct_name YANG DIPERBAIKI")
print("=" * 70)
print()

test_cases = [
    ('Anmeic-fn-0001 1.png',       'anemic',     'Anemic-fn-0001 1.png'),
    ('Anmeic-fn-0001.png',         'anemic',     'Anemic-fn-0001.png'),
    ('Anmeic-fn-001 1.png',        'anemic',     'Anemic-fn-001 1.png'),
    ('Anmeic-fn-0011 (2).png',     'anemic',     'Anemic-fn-0011 (2).png'),
    ('Anmeic-fn-0011 2 (2).png',   'anemic',     'Anemic-fn-0011 2 (2).png'),
    ('Anmeic-fn-0011 2.png',       'anemic',     'Anemic-fn-0011 2.png'),
    ('Anmeic-fn-0011.png',         'anemic',     'Anemic-fn-0011.png'),
    ('Anmeic-fn-0013.png',         'anemic',     'Anemic-fn-0013.png'),
    ('Anrmic-fn-001.png',          'anemic',     'Anemic-fn-001.png'),
    ('anmeic-fn-001.png',          'anemic',     'anemic-fn-001.png'),
    ('Non-anmeic-Fin-001.png',     'nonanemic',  'Non-anemic-Fin-001.png'),
    ('Nonanmeic-Fin-001.png',      'nonanemic',  'Nonanemic-Fin-001.png'),
]

print(f"{'Filename Asli':<32} | {'Hasil Rename':<32} | {'Expected':<32} | Status")
print("-" * 115)

all_pass = True
for filename, label, expected in test_cases:
    result = generate_correct_name_FIXED(filename, label)
    status = "PASS" if result == expected else "FAIL"
    if result != expected:
        all_pass = False
    print(f"{filename:<32} | {result:<32} | {expected:<32} | {status}")

print()
if all_pass:
    print("SEMUA TEST PASS — Fungsi FIXED siap digunakan!")
else:
    print("Ada test yang FAIL — periksa logika fungsi")

print()
print("=" * 70)
print("RE-SCAN DATASET DENGAN FUNGSI YANG DIPERBAIKI")
print("=" * 70)
print()

# Scan ulang semua file di Drive
all_files = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        print(f"Folder tidak ditemukan: {site_path}")
        continue

    for f in sorted(os.listdir(site_path)):
        fp = os.path.join(site_path, f)
        if not os.path.isfile(fp):
            continue

        is_typo, correct_label, typo_word = detect_typo(f)
        # PERBAIKAN: pecah ke beberapa baris supaya tidak terpotong
        if is_typo:
            correct_name = generate_correct_name_FIXED(f, correct_label)
        else:
            correct_name = f

        all_files.append({
            'site': site,
            'filename': f,
            'path': fp,
            'is_typo': is_typo,
            'typo_word': typo_word,
            'correct_label': correct_label,
            'correct_filename': correct_name,
            'needs_rename': is_typo and (f != correct_name)
        })

df_files = pd.DataFrame(all_files)
typo_df = df_files[df_files['is_typo']]
rename_df = df_files[df_files['needs_rename']]

print(f"Total file di-scan: {len(df_files)}")
print(f"File dengan typo: {len(typo_df)}")
print(f"File yang perlu di-rename: {len(rename_df)}")
print()

# Tampilkan contoh rename yang BENAR
print("CONTOH RENAME DENGAN FUNGSI YANG DIPERBAIKI (15 pertama):")
print("-" * 95)
print(f"{'Site':<12} | {'Filename Asli':<35} | {'Filename Benar':<35}")
print("-" * 95)
for _, row in rename_df.head(15).iterrows():
    print(f"{row['site']:<12} | {row['filename']:<35} | {row['correct_filename']:<35}")

if len(rename_df) > 15:
    print(f"  ... dan {len(rename_df) - 15} file lainnya")
print()

# =============================================================================
# VERIFIKASI PENTING: Pastikan -fn- / -FN- / -Fin- tetap ada
# =============================================================================
print("=" * 70)
print("VERIFIKASI: Pastikan '-fn-' / '-FN-' / '-Fin-' tetap ada")
print("=" * 70)
print()

# Cek file rename yang BUKAN mempertahankan pattern
nail_renames = rename_df[rename_df['site'] == 'Nail']
pattern = r'-(fn|FN|Fin)-'
broken = nail_renames[~nail_renames['correct_filename'].str.contains(pattern, regex=True, na=False)]

if len(broken) > 0:
    print(f"MASIH ADA {len(broken)} file Nail yang bagian '-fn-' hilang!")
    print()
    print("Contoh file yang masih bermasalah:")
    for _, row in broken.head(5).iterrows():
        print(f"  {row['filename']}  →  {row['correct_filename']}")
    print()
    print("JANGAN lanjut ke Cell 3! Ada bug yang belum teratasi.")
else:
    print(f"SEMUA {len(nail_renames)} file rename Nail mempertahankan pattern -fn-/-FN-/-Fin-")
    print("  Aman untuk lanjut ke Cell 3.")
print()

# Cek konflik
print("CEK KONFLIK (apakah nama benar sudah dipakai file lain?):")
print("-" * 60)
existing_files_per_site = {}
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if os.path.exists(site_path):
        existing_files_per_site[site] = set(os.listdir(site_path))

conflicts = []
for _, row in rename_df.iterrows():
    if row['correct_filename'] in existing_files_per_site.get(row['site'], set()):
        expected_path = os.path.join(BASE, row['site'], row['correct_filename'])
        if row['path'] != expected_path:
            conflicts.append(row)

if conflicts:
    print(f"DITEMUKAN {len(conflicts)} KONFLIK (nama benar sudah dipakai file lain):")
    print("  File-file ini akan dihapus sebagai duplikat (MD5 identik) di Cell 3.")
    print()
    print("Contoh konflik:")
    for row in conflicts[:5]:
        print(f"  {row['site']}: {row['filename']} → {row['correct_filename']} (sudah ada)")
else:
    print("Tidak ada konflik — semua nama benar belum dipakai file lain")
print()

# =============================================================================
# SIMPAN DAFTAR RENAME (VERSI FIXED)
# =============================================================================
rename_list = rename_df[['site', 'filename', 'correct_filename', 'path']].copy()
rename_list['new_path'] = rename_list.apply(
    lambda r: os.path.join(BASE, r['site'], r['correct_filename']), axis=1
)

rename_csv = os.path.join(OUT, 'daftar_rename_typo_FIXED.csv')
rename_list.to_csv(rename_csv, index=False)
print(f"Daftar rename (FIXED) disimpan: {rename_csv}")
print(f"  Total baris: {len(rename_list)}")
print()

# =============================================================================
# RINGKASAN
# =============================================================================
print("=" * 70)
print("RINGKASAN")
print("=" * 70)
print(f"Total file di Drive:           {len(df_files)}")
print(f"File dengan typo:              {len(typo_df)}")
print(f"File yang akan direname:       {len(rename_df)}")
print()
print("File output:")
print(f"  {rename_csv}")
print()

if all_pass and len(broken) == 0:
    print("SEMUA VERIFIKASI PASS!")
    print()
    print("LANGKAH SELANJUTNYA:")
    print("  1. Pastikan backup lengkap (12,793 file) — SUDAH OK")
    print("  2. Jalankan Cell 3 (rename + hapus duplikat)")
    print("  3. Cell 3 akan otomatis pakai daftar_rename_typo_FIXED.csv")
else:
    print("Ada masalah yang perlu diperbaiki sebelum lanjut ke Cell 3.")

In [ ]:
# Verifikasi: Bandingkan file Nail yang BENAR vs hasil rename BUG
import os

nail_path = '/content/drive/MyDrive/Dataset_Multisite/Nail'
files = sorted(os.listdir(nail_path))

# Cari file Nail yang sudah benar (ada -fn- atau -FN-)
correct_files = [f for f in files if '-fn-' in f.lower() or '-FN-' in f]
# Cari file Nail yang typo
typo_files = [f for f in files if 'anmeic' in f.lower() or 'anrmic' in f.lower()]

print("CONTOH FILE NAIL YANG SUDAH BENAR (format referensi):")
for f in correct_files[:5]:
    print(f"  ✓ {f}")

print()
print("CONTOH FILE NAIL YANG TYPO (perlu diperbaiki):")
for f in typo_files[:5]:
    print(f"  ⚠ {f}")

print()
print("=" * 60)
print("PERHATIKAN:")
print("-" * 60)
print(f"File BENAR punya pattern: -fn- atau -FN-")
print(f"File typo juga punya pattern: -fn- (hanya kata Anmeic-nya yang typo)")
print()
print("Jadi hasil rename yang BENAR harus RETAIN bagian -fn-!")
print("Contoh: Anmeic-fn-001.png → Anemic-fn-001.png (BUKAN Anemic001.png)")

In [ ]:
import os

BASE = '/content/drive/MyDrive/Dataset_Multisite'
# Cari folder backup (cari yang terbaru)
import glob
backups = sorted(glob.glob('/content/drive/MyDrive/Dataset_Multisite_BACKUP_*'))
if backups:
    BACKUP = backups[-1]  # Ambil yang terbaru
else:
    print("Tidak ada folder backup ditemukan")
    BACKUP = None

if BACKUP:
    print(f"Backup folder: {BACKUP}")
    print()
    print("Isi backup:")
    for item in sorted(os.listdir(BACKUP)):
        full = os.path.join(BACKUP, item)
        if os.path.isdir(full):
            count = len(os.listdir(full))
            print(f"  📁 {item} ({count} files)")

    print()
    print("Isi source (BASE):")
    for item in sorted(os.listdir(BASE)):
        full = os.path.join(BASE, item)
        if os.path.isdir(full):
            count = len(os.listdir(full))
            print(f"  📁 {item} ({count} files)")

    # Cek apakah Conjunctiva ada di backup
    backup_conj = os.path.join(BACKUP, 'Conjunctiva')
    if not os.path.exists(backup_conj):
        print()
        print(f"🚨 PERINGATAN: Folder 'Conjunctiva' TIDAK ADA di backup!")
        print(f"   Backup tidak lengkap. HAPUS backup ini dan ulangi Cell 1.")

In [ ]:
# CELL 3 (FIXED): RENAME TYPO + HAPUS DUPLIKAT
# =============================================================================
# Menggunakan daftar_rename_typo_FIXED.csv (bukan yang lama)
# =============================================================================

import os
import re
import hashlib
import json
import pandas as pd
from datetime import datetime

BASE = '/content/drive/MyDrive/Dataset_Multisite'
OUT = '/content/drive/MyDrive/hasil_revisi_v2'

# === STEP A: RENAME TYPO ===
print("=" * 70)
print("STEP A: RENAME FILE TYPO")
print("=" * 70)
print()

rename_csv = os.path.join(OUT, 'daftar_rename_typo_FIXED.csv')

if not os.path.exists(rename_csv):
    raise FileNotFoundError(f"File tidak ditemukan: {rename_csv}. Jalankan Cell 2 dulu.")

rename_df = pd.read_csv(rename_csv)
print(f"File yang akan direname: {len(rename_df)}")
print()

rename_success = 0
rename_skipped_duplicate = 0
rename_skipped_notfound = 0
rename_failed = []

for idx, row in rename_df.iterrows():
    old_path = row['path']
    new_path = row['new_path']
    site = row['site']
    old_name = row['filename']
    new_name = row['correct_filename']

    # Cek apakah file asli masih ada
    if not os.path.exists(old_path):
        rename_skipped_notfound += 1
        continue

    # Cek apakah target sudah ada (konflik)
    if os.path.exists(new_path):
        # Bandingkan MD5
        try:
            with open(old_path, 'rb') as f:
                old_md5 = hashlib.md5(f.read()).hexdigest()
            with open(new_path, 'rb') as f:
                new_md5 = hashlib.md5(f.read()).hexdigest()

            if old_md5 == new_md5:
                # Duplikat identik → hapus file typo
                os.remove(old_path)
                rename_skipped_duplicate += 1
            else:
                # Beda konten → rename dengan suffix _fixed
                if '.' in new_name:
                    base_name, ext = new_name.rsplit('.', 1)
                    new_name_fixed = f"{base_name}_fixed.{ext}"
                else:
                    new_name_fixed = f"{new_name}_fixed"
                new_path_fixed = os.path.join(BASE, site, new_name_fixed)
                os.rename(old_path, new_path_fixed)
                rename_success += 1
        except Exception as e:
            rename_failed.append({'file': old_name, 'error': str(e)})
    else:
        # Tidak ada konflik → rename langsung
        try:
            os.rename(old_path, new_path)
            rename_success += 1
        except Exception as e:
            rename_failed.append({'file': old_name, 'error': str(e)})

    # Progress setiap 200 file
    if (idx + 1) % 200 == 0:
        print(f"  Progres: {idx + 1}/{len(rename_df)}")

print()
print(f"Berhasil rename: {rename_success} file")
print(f"Dihapus (duplikat identik): {rename_skipped_duplicate} file")
print(f"Skip (file tidak ditemukan): {rename_skipped_notfound} file")
if rename_failed:
    print(f"Gagal: {len(rename_failed)} file")
    for f in rename_failed[:5]:
        print(f"  - {f['file']}: {f['error']}")
print()

# === STEP B: HAPUS DUPLIKAT MD5 ===
print("=" * 70)
print("STEP B: HAPUS DUPLIKAT MD5")
print("=" * 70)
print()

def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

# Scan ulang semua file (setelah rename)
all_files_after_rename = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    for f in sorted(os.listdir(site_path)):
        fp = os.path.join(site_path, f)
        if not os.path.isfile(fp):
            continue
        all_files_after_rename.append({
            'site': site,
            'filename': f,
            'path': fp,
            'md5': md5(fp)
        })

df_after = pd.DataFrame(all_files_after_rename)
print(f"Total file setelah rename: {len(df_after)}")

# Identifikasi duplikat MD5 (keep first)
duplicates = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_data = df_after[df_after['site'] == site]
    dup_mask = site_data.duplicated('md5', keep='first')
    duplicates.extend(site_data[dup_mask]['path'].tolist())

print(f"Duplikat MD5 yang akan dihapus: {len(duplicates)}")
print()

deleted_count = 0
for path in duplicates:
    if os.path.exists(path):
        os.remove(path)
        deleted_count += 1

print(f"Dihapus: {deleted_count} file")
print()

# === STEP C: HAPUS SALINAN BERNOMOR ===
print("=" * 70)
print("STEP C: HAPUS SALINAN BERNOMOR (FILE DENGAN TANDA KURUNG)")
print("=" * 70)
print()

numbered_deleted = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    numbered_files = [f for f in os.listdir(site_path) if '(' in f]
    print(f"  {site}: {len(numbered_files)} file berkurung")
    for f in numbered_files:
        fp = os.path.join(site_path, f)
        if os.path.exists(fp):
            os.remove(fp)
            numbered_deleted += 1

print()
print(f"Dihapus: {numbered_deleted} file")
print()

# === STEP D: HAPUS FILE VARIAN (FV/RA) ===
print("=" * 70)
print("STEP D: HAPUS FILE VARIAN (AKHIRAN FV/RA)")
print("=" * 70)
print()

variant_deleted = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    variant_files = [f for f in os.listdir(site_path) if re.search(r'(FV|RA)\.[A-Za-z]+$', f)]
    print(f"  {site}: {len(variant_files)} file varian")
    for f in variant_files:
        fp = os.path.join(site_path, f)
        if os.path.exists(fp):
            os.remove(fp)
            variant_deleted += 1

print()
print(f"Dihapus: {variant_deleted} file")
print()

# === SIMPAN STATISTIK UNTUK CELL 4 ===
stats = {
    'rename_success': rename_success,
    'rename_skipped_duplicate': rename_skipped_duplicate,
    'rename_skipped_notfound': rename_skipped_notfound,
    'duplicate_md5_deleted': deleted_count,
    'numbered_deleted': numbered_deleted,
    'variant_deleted': variant_deleted,
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

stats_path = os.path.join(OUT, 'stats_pembersihan.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)

# === RINGKASAN ===
print("=" * 70)
print("RINGKASAN PEMBERSIHAN")
print("=" * 70)
print(f"Berhasil rename:                {rename_success} file")
print(f"Dihapus (duplikat identik):     {rename_skipped_duplicate} file")
print(f"Dihapus (duplikat MD5):         {deleted_count} file")
print(f"Dihapus (salinan bernomor):     {numbered_deleted} file")
print(f"Dihapus (file varian FV/RA):    {variant_deleted} file")
print()

# Hitung file final
final_counts = {}
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if os.path.exists(site_path):
        final_counts[site] = len([f for f in os.listdir(site_path)
                                  if os.path.isfile(os.path.join(site_path, f))])

print("DISTRIBUSI FILE FINAL DI DRIVE:")
for site, count in final_counts.items():
    print(f"  {site}: {count} file")
print(f"  TOTAL: {sum(final_counts.values())} file")
print()
print(f"Statistik disimpan: {stats_path}")

In [ ]:
# RESTORE DATASET DARI BACKUP
# =============================================================================
# Karena Step B menghapus terlalu banyak file (termasuk file asli),
# kita perlu restore dataset dari backup dan menjalankan ulang dengan
# logika yang diperbaiki.
# =============================================================================

import os
import shutil
import glob

BASE = '/content/drive/MyDrive/Dataset_Multisite'

# Cari folder backup terbaru
backups = sorted(glob.glob('/content/drive/MyDrive/Dataset_Multisite_BACKUP_*'))
if not backups:
    raise FileNotFoundError("Tidak ada folder backup ditemukan!")

BACKUP = backups[-1]
print(f"Backup terbaru: {BACKUP}")
print()

# Cek isi backup
print("Isi backup:")
for item in sorted(os.listdir(BACKUP)):
    full = os.path.join(BACKUP, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {item}: {count} files")
print()

# Cek isi BASE (yang sekarang rusak)
print("Isi BASE (yang akan direstore):")
for item in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {item}: {count} files")
print()

# Konfirmasi
print("=" * 70)
print("PERINGATAN: Operasi ini akan MENGHAPUS folder BASE yang sekarang")
print(f"dan MENGGANTI dengan isi dari backup: {BACKUP}")
print()
print("Backup Anda tetap aman (tidak dihapus).")
print()

# Hapus folder BASE yang rusak
print("Menghapus folder BASE yang rusak...")
shutil.rmtree(BASE)
print("✓ Folder BASE dihapus")
print()

# Copy dari backup ke BASE
print(f"Copy dari backup ke BASE... (5-10 menit)")
shutil.copytree(BACKUP, BASE)
print("✓ Restore selesai!")
print()

# Verifikasi
print("Verifikasi hasil restore:")
for item in sorted(os.listdir(BASE)):
    full = os.path.join(BASE, item)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f"  {item}: {count} files")

total = sum(len(os.listdir(os.path.join(BASE, s)))
            for s in ['Palm', 'Nail', 'Conjunctiva']
            if os.path.exists(os.path.join(BASE, s)))
print(f"  TOTAL: {total} files")
print()
print("✓ Dataset sudah restore ke kondisi awal (12,793 files)")
print("  Sekarang jalankan Cell 3 yang sudah DIPERBAIKI")

In [ ]:
# CELL 3 (FIXED v2): RENAME TYPO + HAPUS DUPLIKAT DENGAN PRIORITAS
# =============================================================================
# Bug yang diperbaiki:
#   Versi lama: keep first berdasarkan sorted() → file varian dipertahankan,
#               file asli dihapus (karena urutan sorted meletakkan varian dulu)
#
#   Versi FIXED: prioritas keep file ASLI (tanpa suffix),
#               hapus file varian (spasi+angka, kurung, FV/RA) jika MD5 sama
# =============================================================================

import os
import re
import hashlib
import json
import shutil
import pandas as pd
from datetime import datetime

BASE = '/content/drive/MyDrive/Dataset_Multisite'
OUT = '/content/drive/MyDrive/hasil_revisi_v2'

# === STEP A: RENAME TYPO (sama seperti sebelumnya) ===
print("=" * 70)
print("STEP A: RENAME FILE TYPO")
print("=" * 70)
print()

rename_csv = os.path.join(OUT, 'daftar_rename_typo_FIXED.csv')
if not os.path.exists(rename_csv):
    raise FileNotFoundError(f"File tidak ditemukan: {rename_csv}")

rename_df = pd.read_csv(rename_csv)
print(f"File yang akan direname: {len(rename_df)}")
print()

rename_success = 0
rename_skipped_duplicate = 0
rename_skipped_notfound = 0

for idx, row in rename_df.iterrows():
    old_path = row['path']
    new_path = row['new_path']
    old_name = row['filename']
    new_name = row['correct_filename']

    if not os.path.exists(old_path):
        rename_skipped_notfound += 1
        continue

    if os.path.exists(new_path):
        try:
            with open(old_path, 'rb') as f:
                old_md5 = hashlib.md5(f.read()).hexdigest()
            with open(new_path, 'rb') as f:
                new_md5 = hashlib.md5(f.read()).hexdigest()

            if old_md5 == new_md5:
                os.remove(old_path)
                rename_skipped_duplicate += 1
            else:
                if '.' in new_name:
                    base_n, ext = new_name.rsplit('.', 1)
                    new_name_fixed = f"{base_n}_fixed.{ext}"
                else:
                    new_name_fixed = f"{new_name}_fixed"
                new_path_fixed = os.path.join(BASE, row['site'], new_name_fixed)
                os.rename(old_path, new_path_fixed)
                rename_success += 1
        except Exception:
            pass
    else:
        try:
            os.rename(old_path, new_path)
            rename_success += 1
        except Exception:
            pass

    if (idx + 1) % 200 == 0:
        print(f"  Progres: {idx + 1}/{len(rename_df)}")

print()
print(f"Berhasil rename: {rename_success}")
print(f"Dihapus (duplikat identik): {rename_skipped_duplicate}")
print(f"Skip (tidak ditemukan): {rename_skipped_notfound}")
print()

# === STEP B (FIXED): HAPUS DUPLIKAT MD5 DENGAN PRIORITAS KEEP FILE ASLI ===
print("=" * 70)
print("STEP B (FIXED): HAPUS DUPLIKAT MD5 DENGAN PRIORITAS")
print("=" * 70)
print()

def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def file_priority(filename):
    """
    Tentukan prioritas file (lebih kecil = lebih diprioritaskan untuk keep).

    Prioritas:
      1 = FILE ASLI (tanpa suffix, tanpa kurung, tanpa spasi+angka)
      2 = File varian FV/RA (karena ini augmentasi intentional)
      3 = File dengan spasi+angka (mis. "Anemic-001 1.png")
      4 = File berkurung (mis. "Anemic-001 (1).png")
    """
    if '(' in filename:
        return 4
    if re.search(r'\s\d+\.[A-Za-z]+$', filename):
        return 3
    if re.search(r'(FV|RA)\.[A-Za-z]+$', filename):
        return 2
    return 1  # File asli (prioritas tertinggi)

# Scan semua file setelah rename
all_files = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    for f in sorted(os.listdir(site_path)):
        fp = os.path.join(site_path, f)
        if not os.path.isfile(fp):
            continue
        all_files.append({
            'site': site,
            'filename': f,
            'path': fp,
            'md5': md5(fp),
            'priority': file_priority(f)
        })

df_files = pd.DataFrame(all_files)
print(f"Total file setelah rename: {len(df_files)}")
print()

# Distribusi prioritas
print("Distribusi prioritas file:")
priority_names = {1: 'File asli (tanpa suffix)', 2: 'Varian FV/RA',
                  3: 'Spasi+angka', 4: 'Berkurung'}
for p, name in priority_names.items():
    count = (df_files['priority'] == p).sum()
    print(f"  Prioritas {p} ({name}): {count} file")
print()

# Untuk setiap grup MD5, keep file dengan prioritas TERKECIL (paling asli)
# Jika ada tie, keep yang urutan sorted-nya pertama
df_files = df_files.sort_values(['site', 'md5', 'priority', 'filename'])
# Tandai keep = False untuk semua kecuali first occurrence per (site, md5)
df_files['keep'] = ~df_files.duplicated(subset=['site', 'md5'], keep='first')

# Statistik
to_delete = df_files[~df_files['keep']]
print(f"File yang akan dihapus (duplikat MD5): {len(to_delete)}")
print()

# Distribusi prioritas yang dihapus vs di-keep
print("Distribusi prioritas yang DI-KEEP:")
keep_dist = df_files[df_files['keep']]['priority'].value_counts().sort_index()
for p, count in keep_dist.items():
    print(f"  Prioritas {p} ({priority_names[p]}): {count} file")
print()

print("Distribusi prioritas yang DIHAPUS:")
delete_dist = df_files[~df_files['keep']]['priority'].value_counts().sort_index()
for p, count in delete_dist.items():
    print(f"  Prioritas {p} ({priority_names[p]}): {count} file")
print()

# Hapus file duplikat
deleted_count = 0
for _, row in to_delete.iterrows():
    if os.path.exists(row['path']):
        os.remove(row['path'])
        deleted_count += 1

print(f"Dihapus: {deleted_count} file")
print()

# === STEP C: HAPUS SALINAN BERNOMOR (jika masih ada) ===
print("=" * 70)
print("STEP C: HAPUS SALINAN BERNOMOR")
print("=" * 70)
print()

numbered_deleted = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    numbered_files = [f for f in os.listdir(site_path) if '(' in f]
    print(f"  {site}: {len(numbered_files)} file berkurung")
    for f in numbered_files:
        fp = os.path.join(site_path, f)
        if os.path.exists(fp):
            os.remove(fp)
            numbered_deleted += 1

print()
print(f"Dihapus: {numbered_deleted} file")
print()

# === STEP D: HAPUS FILE VARIAN FV/RA ===
print("=" * 70)
print("STEP D: HAPUS FILE VARIAN FV/RA")
print("=" * 70)
print()

variant_deleted = 0
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    variant_files = [f for f in os.listdir(site_path) if re.search(r'(FV|RA)\.[A-Za-z]+$', f)]
    print(f"  {site}: {len(variant_files)} file varian")
    for f in variant_files:
        fp = os.path.join(site_path, f)
        if os.path.exists(fp):
            os.remove(fp)
            variant_deleted += 1

print()
print(f"Dihapus: {variant_deleted} file")
print()

# === SIMPAN STATISTIK ===
stats = {
    'rename_success': rename_success,
    'rename_skipped_duplicate': rename_skipped_duplicate,
    'duplicate_md5_deleted': deleted_count,
    'numbered_deleted': numbered_deleted,
    'variant_deleted': variant_deleted,
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

stats_path = os.path.join(OUT, 'stats_pembersihan_v2.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)

# === RINGKASAN ===
print("=" * 70)
print("RINGKASAN PEMBERSIHAN (FIXED v2)")
print("=" * 70)
print(f"Berhasil rename:                {rename_success} file")
print(f"Dihapus (duplikat identik):     {rename_skipped_duplicate} file")
print(f"Dihapus (duplikat MD5):         {deleted_count} file")
print(f"Dihapus (salinan bernomor):     {numbered_deleted} file")
print(f"Dihapus (file varian FV/RA):    {variant_deleted} file")
print()

# Hitung file final
final_counts = {}
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if os.path.exists(site_path):
        final_counts[site] = len([f for f in os.listdir(site_path)
                                  if os.path.isfile(os.path.join(site_path, f))])

print("DISTRIBUSI FILE FINAL DI DRIVE:")
for site, count in final_counts.items():
    print(f"  {site}: {count} file")
print(f"  TOTAL: {sum(final_counts.values())} file")
print()

# Perbandingan dengan ekspektasi
print("PERBANDINGAN DENGAN EKSEPTASI:")
expected = {'Palm': 526, 'Nail': 556, 'Conjunctiva': 547}
for site in ['Palm', 'Nail', 'Conjunctiva']:
    actual = final_counts.get(site, 0)
    exp = expected[site]
    diff = actual - exp
    status = "✓" if abs(diff) <= 10 else "⚠"
    print(f"  {site}: actual={actual}, expected={exp}, selisih={diff:+} {status}")

In [ ]:
# CELL 4 (FIXED): VERIFIKASI + LAPORAN AKHIR
# =============================================================================

import os
import re
import json
import glob
import hashlib
import difflib
import pandas as pd
from datetime import datetime

BASE = '/content/drive/MyDrive/Dataset_Multisite'
OUT = '/content/drive/MyDrive/hasil_revisi_v2'
TYPO_THRESHOLD = 0.75

def extract_label_text(name):
    base = name.rsplit('.', 1)[0] if '.' in name else name
    match = re.match(r'^([A-Za-z\-]+)', base)
    if not match:
        return ''
    return match.group(1).lower().replace('-', '').replace(' ', '')

def similarity(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("=" * 70)
print("VERIFIKASI DATASET BERSIH")
print("=" * 70)
print()

# === STEP A: CEK TYPO TERSISA ===
print("STEP A: CEK TYPO TERSISA")
print("-" * 60)

remaining_typos = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    for f in sorted(os.listdir(site_path)):
        fp = os.path.join(site_path, f)
        if not os.path.isfile(fp):
            continue
        text = extract_label_text(f)
        if len(text) < 3:
            continue
        # Cek apakah file ini typo
        is_typo = False
        if not text.startswith('anemic') and not text.startswith('nonanemic'):
            n = min(len(text), 6)
            if similarity(text[:n], 'anemic') >= TYPO_THRESHOLD:
                is_typo = True
            n = min(len(text), 9)
            if similarity(text[:n], 'nonanemic') >= TYPO_THRESHOLD:
                is_typo = True
        if is_typo:
            remaining_typos.append({'site': site, 'filename': f})

if remaining_typos:
    print(f"MASIH ADA {len(remaining_typos)} TYPO TERSISA:")
    for t in remaining_typos[:10]:
        print(f"  {t['site']}/{t['filename']}")
else:
    print("TIDAK ADA TYPO LAGI — semua file punya nama yang benar!")
print()

# === STEP B: CEK DUPLIKAT MD5 ===
print("STEP B: CEK DUPLIKAT MD5 TERSISA")
print("-" * 60)

remaining_duplicates = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    md5s = {}
    for f in os.listdir(site_path):
        fp = os.path.join(site_path, f)
        if not os.path.isfile(fp):
            continue
        h = md5(fp)
        if h in md5s:
            remaining_duplicates.append({
                'site': site,
                'file1': md5s[h],
                'file2': f
            })
        else:
            md5s[h] = f

if remaining_duplicates:
    print(f"MASIH ADA {len(remaining_duplicates)} DUPLIKAT MD5:")
    for d in remaining_duplicates[:10]:
        print(f"  {d['site']}: {d['file1']} == {d['file2']}")
else:
    print("TIDAK ADA DUPLIKAT MD5 — semua file unik!")
print()

# === STEP C: CEK FILE BERNOMOR / VARIAN ===
print("STEP C: CEK FILE BERNOMOR / VARIAN")
print("-" * 60)

remaining_numbered = []
remaining_variant = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    for f in os.listdir(site_path):
        if '(' in f:
            remaining_numbered.append({'site': site, 'filename': f})
        if re.search(r'(FV|RA)\.[A-Za-z]+$', f):
            remaining_variant.append({'site': site, 'filename': f})

if remaining_numbered:
    print(f"Masih ada {len(remaining_numbered)} file berkurung")
else:
    print("TIDAK ADA file berkurung (salinan bernomor)")

if remaining_variant:
    print(f"Masih ada {len(remaining_variant)} file varian (FV/RA)")
else:
    print("TIDAK ADA file varian (FV/RA)")
print()

# === STEP D: DISTRIBUSI FINAL ===
print("STEP D: DISTRIBUSI FILE FINAL DI DRIVE")
print("-" * 60)

final_data = []
for site in ['Palm', 'Nail', 'Conjunctiva']:
    site_path = os.path.join(BASE, site)
    if not os.path.exists(site_path):
        continue
    files = [f for f in os.listdir(site_path) if os.path.isfile(os.path.join(site_path, f))]
    anemic = sum(1 for f in files if f.lower().replace(' ', '').replace('-', '').startswith('anemic'))
    non_anemic = sum(1 for f in files if f.lower().replace(' ', '').replace('-', '').startswith('nonanemic'))
    other = len(files) - anemic - non_anemic
    final_data.append({
        'Site': site,
        'Anemic': anemic,
        'Non-Anemic': non_anemic,
        'Other': other,
        'Total': len(files)
    })

final_df = pd.DataFrame(final_data)
print(final_df.to_string(index=False))
print()

total_anemic = final_df['Anemic'].sum()
total_non = final_df['Non-Anemic'].sum()
total = final_df['Total'].sum()
print(f"TOTAL: {total} file (Anemic: {total_anemic}, Non-Anemic: {total_non})")
print(f"Prevalensi anemic: {100 * total_anemic / total:.1f}%")
print()

# === STEP E: SIMPAN LAPORAN ===
print("STEP E: SIMPAN LAPORAN")
print("-" * 60)

# Load stats dari Cell 3
stats_path = os.path.join(OUT, 'stats_pembersihan.json')
if os.path.exists(stats_path):
    with open(stats_path, 'r') as f:
        stats = json.load(f)
else:
    stats = {}

# Cari folder backup
backups = sorted(glob.glob('/content/drive/MyDrive/Dataset_Multisite_BACKUP_*'))
backup_path = backups[-1] if backups else 'Tidak ditemukan'

report = {
    'cleaning_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'operations_performed': {
        'typo_rename_count': stats.get('rename_success', 'N/A'),
        'duplicate_identik_deleted': stats.get('rename_skipped_duplicate', 'N/A'),
        'duplicate_md5_deleted': stats.get('duplicate_md5_deleted', 'N/A'),
        'numbered_copies_deleted': stats.get('numbered_deleted', 'N/A'),
        'variant_files_deleted': stats.get('variant_deleted', 'N/A'),
    },
    'final_distribution': final_df.to_dict('records'),
    'totals': {
        'total_anemic': int(total_anemic),
        'total_non_anemic': int(total_non),
        'total_files': int(total),
        'prevalence_anemic_pct': round(100 * total_anemic / total, 2)
    },
    'verification': {
        'remaining_typo': len(remaining_typos),
        'remaining_duplicates_md5': len(remaining_duplicates),
        'remaining_numbered_files': len(remaining_numbered),
        'remaining_variant_files': len(remaining_variant)
    },
    'backup_location': backup_path
}

report_path = os.path.join(OUT, 'laporan_pembersihan_drive.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f"Laporan disimpan: {report_path}")
print()

# === LAPORAN AKHIR ===
print("=" * 70)
print("LAPORAN AKHIR")
print("=" * 70)
print(f"Tanggal: {report['cleaning_date']}")
print()
print("OPERASI YANG DILAKUKAN:")
print(f"  - Rename typo:                {stats.get('rename_success', 'N/A')} file")
print(f"  - Hapus duplikat identik:     {stats.get('rename_skipped_duplicate', 'N/A')} file")
print(f"  - Hapus duplikat MD5:         {stats.get('duplicate_md5_deleted', 'N/A')} file")
print(f"  - Hapus salinan bernomor:     {stats.get('numbered_deleted', 'N/A')} file")
print(f"  - Hapus file varian (FV/RA):  {stats.get('variant_deleted', 'N/A')} file")
print()
print("DISTRIBUSI FINAL:")
for d in final_df.to_dict('records'):
    print(f"  {d['Site']:<12}: {d['Anemic']} anemic, {d['Non-Anemic']} non-anemic, total {d['Total']}")
print()
print(f"Total: {total} file")
print(f"Prevalensi anemic: {100 * total_anemic / total:.1f}%")
print()
print("VERIFIKASI:")
print(f"  Typo tersisa:              {len(remaining_typos)}")
print(f"  Duplikat MD5 tersisa:      {len(remaining_duplicates)}")
print(f"  File berkurung tersisa:    {len(remaining_numbered)}")
print(f"  File varian tersisa:       {len(remaining_variant)}")
print()
print(f"Backup: {backup_path}")
print()
if (len(remaining_typos) == 0 and len(remaining_duplicates) == 0
    and len(remaining_numbered) == 0 and len(remaining_variant) == 0):
    print("SEMUA VERIFIKASI PASS — Dataset siap digunakan!")
else:
    print("PERINGATAN: Ada masalah yang belum teratasi. Cek di atas.")

In [ ]:
# CELL 5 (FIXED): TEST PARSER LAMA DENGAN DATASET BERSIH
# =============================================================================
# Setelah Drive dibersihkan, parser LAMA (yang tidak tangkap typo)
# sekarang akan bisa membaca SEMUA file karena nama sudah diperbaiki.
# =============================================================================

import os
import pandas as pd

BASE = '/content/drive/MyDrive/Dataset_Multisite'
SITES = {'Palm': 0, 'Nail': 1, 'Conjunctiva': 2}

def parse_label_lama(name):
    """Parser LAMA dari Notebook 17 — sebelum pembersihan."""
    if '(' in name:
        return None
    c = name.lower().replace(' ', '')
    if c.startswith('non-anemic'):
        return 0
    if c.startswith('anemic'):
        return 1
    return None

print("=" * 70)
print("TEST PARSER LAMA DENGAN DATASET YANG SUDAH DIBERSIHKAN")
print("=" * 70)
print()

rows = []
unrecognized = []
for site, sid in SITES.items():
    d = os.path.join(BASE, site)
    if not os.path.exists(d):
        print(f"Folder tidak ditemukan: {d}")
        continue
    for n in sorted(os.listdir(d)):
        fp = os.path.join(d, n)
        if not os.path.isfile(fp):
            continue
        lab = parse_label_lama(n)
        if lab is not None:
            rows.append(dict(site=site, site_id=sid, file=n, path=fp, label=lab))
        else:
            unrecognized.append({'site': site, 'filename': n})

df = pd.DataFrame(rows)

print(f"File terbaca parser LAMA: {len(df)}")
print(f"File tidak terbaca: {len(unrecognized)}")
print()

if unrecognized:
    print("FILE TIDAK TERBACA (harusnya 0 setelah pembersihan):")
    for u in unrecognized[:10]:
        print(f"  {u['site']}/{u['filename']}")
    if len(unrecognized) > 10:
        print(f"  ... dan {len(unrecognized) - 10} file lainnya")
else:
    print("SEMUA FILE TERBACA PARSER LAMA!")
    print("  Parser lama sekarang work karena nama sudah diperbaiki.")
print()

print("DISTRIBUSI (parser lama, dataset bersih):")
print(df.groupby('site').label.agg(total='count', anemic='sum'))
print()

print("=" * 70)
print("KESIMPULAN")
print("=" * 70)
print()
print("Dataset di Google Drive sudah sepenuhnya bersih:")
print("  - Semua typo sudah direname ke nama yang benar")
print("  - Semua duplikat sudah dihapus")
print("  - Semua salinan bernomor sudah dihapus")
print("  - Semua file varian (FV/RA) sudah dihapus")
print()
print("Parser LAMA sekarang bisa membaca semua file")
print("(karena nama sudah diperbaiki, tidak perlu smart parser lagi)")
print()
print("LANGKAH SELANJUTNYA:")
print("  1. Run ulang Notebook 16 (audit) untuk verifikasi dataset bersih")
print("  2. Run ulang Notebook 17 (pooling) untuk dapat hasil eksperimen baru")
print("  3. Run ulang semua notebook lain (18-25) dengan parser lama")
print("  4. Update paper dengan angka baru")
print("  5. Catat di paper: 'Dataset telah dibersihkan dari typo dan duplikat'")

In [ ]:
# INVESTIGASI: Cek file non-anemic di Nail
# =============================================================================

import os
import re

BASE = '/content/drive/MyDrive/Dataset_Multisite/Nail'
files = sorted(os.listdir(BASE))

# Klasifikasi detail
non_anemic_fin = []      # Format asli: Non-anemic-Fin-XXX.png
non_anemic_other = []    # Format lain: mungkin hasil rename
anemic_fn = []           # Format asli: Anemic-FN-XXX.png
anemic_fn_lower = []     # Format: Anemic-fn-XXX.png (dari rename typo)
anemic_other = []        # Format lain

for f in files:
    c = f.lower().replace(' ', '')
    if c.startswith('non-anemic'):
        if '-fin-' in c or '-Fin-' in f:
            non_anemic_fin.append(f)
        else:
            non_anemic_other.append(f)
    elif c.startswith('anemic'):
        if '-FN-' in f:
            anemic_fn.append(f)
        elif '-fn-' in f:
            anemic_fn_lower.append(f)
        else:
            anemic_other.append(f)

print("=" * 70)
print("KLASIFIKASI DETAIL FILE NAIL")
print("=" * 70)
print()
print(f"ANEMIC (format asli -FN-):     {len(anemic_fn)} file")
print(f"ANEMIC (format -fn- lowercase): {len(anemic_fn_lower)} file (dari rename typo)")
print(f"ANEMIC (format lain):           {len(anemic_other)} file")
print(f"NON-ANEMIC (format -Fin-):      {len(non_anemic_fin)} file")
print(f"NON-ANEMIC (format lain):       {len(non_anemic_other)} file")
print()

if non_anemic_other:
    print("FILE NON-ANEMIC DENGAN FORMAT LAIN (perlu investigasi):")
    for f in non_anemic_other[:20]:
        print(f"  {f}")
    if len(non_anemic_other) > 20:
        print(f"  ... dan {len(non_anemic_other) - 20} file lainnya")
    print()

if anemic_other:
    print("FILE ANEMIC DENGAN FORMAT LAIN:")
    for f in anemic_other[:20]:
        print(f"  {f}")
    if len(anemic_other) > 20:
        print(f"  ... dan {len(anemic_other) - 20} file lainnya")
print()

# Cek apakah ada file "Non-anemic" yang sebelumnya typo
# (mis. Non-anmeic → Non-anemic)
print("CEK FILE NON-ANEMIC YANG MEMILIKI POLA UNIK:")
for f in non_anemic_other[:10]:
    # Cek MD5 untuk verifikasi
    import hashlib
    fp = os.path.join(BASE, f)
    with open(fp, 'rb') as fh:
        h = hashlib.md5(fh.read()).hexdigest()
    print(f"  {f} (MD5: {h[:8]}...)")

In [ ]:
# CEK LANGSUNG DARI SERVER GOOGLE DRIVE (bypass cache device)
# =============================================================================

import os

BASE = '/content/drive/MyDrive/Dataset_Multisite/Nail'
files = sorted(os.listdir(BASE))

print(f"Total file di Nail (server Google): {len(files)}")
print()

# Cek file non-anemic
non_anemic = [f for f in files if f.lower().replace(' ', '').startswith('non-anemic')]
anemic = [f for f in files if f.lower().replace(' ', '').startswith('anemic')]
other = [f for f in files if f not in non_anemic and f not in anemic]

print(f"Anemic:     {len(anemic)} file")
print(f"Non-Anemic: {len(non_anemic)} file")
print(f"Other:      {len(other)} file")
print()

print("CONTOH FILE NON-ANEMIC DI NAIL (10 pertama):")
for f in non_anemic[:10]:
    print(f"  {f}")
print()

print("CONTOH FILE ANEMIC DI NAIL (10 pertama):")
for f in anemic[:10]:
    print(f"  {f}")
print()

# Cek apakah ada file "Non-anemic" yang baru direname
# (bukan "Non-anemic-Fin-XXX.png" format original)
print("FILE NON-ANEMIC DENGAN NAMA UNIK (bukan format Fin):")
unique_non_anemic = [f for f in non_anemic if '-Fin-' not in f and '-fin-' not in f]
print(f"  Jumlah: {len(unique_non_anemic)}")
for f in unique_non_anemic[:10]:
    print(f"  {f}")